# MergeSortBottomUp (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_cpp.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `MergeSortBottomUp`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [1]:
%%writefile bottom_up.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // confronti fra elementi

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r) del capitolo 17
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;
    int n2 = r - m;
    vector<int> L(n1), R(n2);
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

void MergeSort(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSort(a, l, m);
        MergeSort(a, m + 1, r);
        Merge(a, l, m, r);
    }
}

// MergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione
void MergeSortBottomUp(int *a, int n)
{
    int width = 1;
    while (width < n)
    {
        int l = 0;
        while (l + width < n)                   // pseudocodice: l + width <= n
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            l = l + 2 * width;
        }
        width = 2 * width;
    }
}

void MergeSortBottomUpTrace(int *a, int n)
{
    int width = 1;
    int passes = 0;
    while (width < n)
    {
        int l = 0;
        string parts;
        while (l + width < n)
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            if (!parts.empty()) parts += ", ";
            parts += "a[" + to_string(l) + ".." + to_string(m) + "] con a[" + to_string(m + 1) + ".." + to_string(r) + "]";
            l = l + 2 * width;
        }
        passes++;
        printf("width=%d: merge %s => %s\n", width, parts.c_str(), Fmt(a, n).c_str());
        width = 2 * width;
    }
    printf("passate = %d\n", passes);
}

// InsertionSort(a, n) (capitolo 3), con il conteggio dei confronti
void InsertionSort(int *a, int n)
{
    for (int j = 1; j < n; j++)
    {
        int key = a[j];
        int i = j - 1;
        while (i >= 0)
        {
            comparisons++;
            if (a[i] > key)
            {
                a[i + 1] = a[i];
                i = i - 1;
            }
            else
            {
                break;
            }
        }
        a[i + 1] = key;
    }
}

// Generatore congruenziale lineare: stessi numeri in Python, C++ e Java
vector<int> RandomArray(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = 1 + (int)(x % 1000000ULL);
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {5, 2, 4, 6, 1, 3};
    printf("a = %s   n = %d\n", Fmt(a, 6).c_str(), 6);
    MergeSortBottomUpTrace(a, 6);

    printf("\n");
    printf("%6s %14s %10s %9s  confronti\n", "n", "InsertionSort", "MergeSort", "BottomUp");
    vector<array<double, 4>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> base = RandomArray(n);
        vector<int> x1 = base, x2 = base, x3 = base;
        comparisons = 0;
        double t0 = NowMs(); InsertionSort(x1.data(), n); double t1 = NowMs() - t0;
        long long c1 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSort(x2.data(), 0, n - 1); double t2 = NowMs() - t0;
        long long c2 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSortBottomUp(x3.data(), n); double t3 = NowMs() - t0;
        long long c3 = comparisons;
        vector<int> ord = base;
        sort(ord.begin(), ord.end());
        const char *ok = (x1 == x2 && x2 == x3 && x3 == ord) ? "OK" : "ERROR";
        printf("%6d %14lld %10lld %9lld  %s\n", n, c1, c2, c3, ok);
        times.push_back({(double)n, t1, t2, t3});
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (auto &t : times)
    {
        printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms\n", (int)t[0], t[1], t[2], t[3]);
    }
    return 0;
}
#endif

Writing bottom_up.cpp


In [2]:
!g++ -std=c++17 -O2 -o bottom_up bottom_up.cpp && ./bottom_up

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort      0.196 ms   MergeSort    0.128 ms   MergeSortBottomUp    0.114 ms
n=2000   InsertionSort      0.824 ms   MergeSort    0.254 ms   MergeSortBottomUp    0.227 ms
n=4000   InsertionSort      3.786 ms   MergeSort    0.569 ms   MergeSortBottomUp    0.475 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [3]:
%%writefile prova_bottom_up.cpp
#define NOMAIN
#include "bottom_up.cpp"

int main()
{
    // Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
    int n = 8000;
    vector<int> base = RandomArray(n);
    vector<int> x1 = base, x2 = base;
    comparisons = 0;
    InsertionSort(x1.data(), n);
    long long c1 = comparisons;
    comparisons = 0;
    MergeSortBottomUp(x2.data(), n);
    printf("%lld %lld %s\n", c1, comparisons, x1 == x2 ? "True" : "False");
    return 0;
}

Writing prova_bottom_up.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_bottom_up prova_bottom_up.cpp && ./prova_bottom_up

16115652 93756 True


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).